# Segment 4 - Product & Returns Performance Analysis

## Analysis 10 — Product Category Revenue Analysis
### Business question: 
### Which product categories generated the highest merchandise revenue from completed orders, and how many units were sold in each category?

In [0]:
SELECT
    p.category,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(oi.quantity) AS units_sold,
    ROUND(SUM(oi.quantity * oi.unit_price), 2)
        AS total_revenue
FROM retailops360.source.orders AS o
INNER JOIN retailops360.source.order_items AS oi
    ON o.order_id = oi.order_id
INNER JOIN retailops360.source.products AS p
    ON oi.product_id = p.product_id
WHERE o.order_status = 'Completed'
GROUP BY p.category
ORDER BY total_revenue DESC;

category,total_orders,units_sold,total_revenue
Sports,10138,22135,13705679.00
Books,9996,21637,13345858.00
Home & Kitchen,10207,22194,13004014.00
Beauty,9970,21742,12987796.00
Electronics,9984,21761,12954322.00
Office,10080,21867,12935588.00
Grocery,10129,22027,12884324.00
Fashion,10047,21858,12649818.00


## Analysis 11 — Products Without Sales Analysis
### Business question: 
### Which products have never appeared in any customer order, potentially indicating low demand or inactive inventory?

In [0]:
SELECT
    p.product_id,
    p.product_name,
    p.category,
    p.list_price
FROM retailops360.source.products AS p
LEFT ANTI JOIN retailops360.source.order_items AS oi
    ON p.product_id = oi.product_id
ORDER BY p.category, p.product_name;

product_id,product_name,category,list_price
2308,Beauty Item 2308,Beauty,553.00
2316,Beauty Item 2316,Beauty,950.00
2324,Beauty Item 2324,Beauty,239.00
2332,Beauty Item 2332,Beauty,1127.00
2340,Beauty Item 2340,Beauty,632.00
2348,Beauty Item 2348,Beauty,1126.00
2356,Beauty Item 2356,Beauty,264.00
2364,Beauty Item 2364,Beauty,1180.00
2372,Beauty Item 2372,Beauty,458.00
2380,Beauty Item 2380,Beauty,774.00


## Analysis 12 — Product Return Rate Analysis
### Business question: 
### Which products have the highest return rates relative to the units sold, and how do those rates vary by product category?

In [0]:
WITH product_sales AS (
    SELECT
        oi.product_id,
        COUNT(DISTINCT oi.order_id) AS orders_with_product
    FROM retailops360.source.order_items AS oi
    INNER JOIN retailops360.source.orders AS o
        ON oi.order_id = o.order_id
    WHERE o.order_status = 'Completed'
    GROUP BY oi.product_id
),

product_returns AS (
    SELECT
        r.product_id,
        COUNT(DISTINCT r.order_id) AS returned_orders
    FROM retailops360.source.returns AS r
    INNER JOIN retailops360.source.orders AS o
        ON r.order_id = o.order_id
    WHERE r.return_status = 'Approved'
      AND o.order_status = 'Completed'
    GROUP BY r.product_id
)

SELECT
    p.product_id,
    p.product_name,
    p.category,
    s.orders_with_product,
    COALESCE(r.returned_orders, 0) AS returned_orders,
    ROUND(
        COALESCE(r.returned_orders, 0) * 100.0
        / NULLIF(s.orders_with_product, 0),
        2
    ) AS return_rate_pct

FROM retailops360.source.products AS p

INNER JOIN product_sales AS s
    ON p.product_id = s.product_id

LEFT JOIN product_returns AS r
    ON p.product_id = r.product_id

ORDER BY
    return_rate_pct DESC,
    s.orders_with_product DESC;

product_id,product_name,category,orders_with_product,returned_orders,return_rate_pct
814,Books Item 0814,Books,26,3,11.54
1165,Sports Item 1165,Sports,30,3,10.00
608,Office Item 0608,Office,32,3,9.38
1099,Fashion Item 1099,Fashion,32,3,9.38
1290,Home & Kitchen Item 1290,Home & Kitchen,32,3,9.38
1009,Electronics Item 1009,Electronics,33,3,9.09
780,Beauty Item 0780,Beauty,34,3,8.82
1351,Grocery Item 1351,Grocery,35,3,8.57
1458,Home & Kitchen Item 1458,Home & Kitchen,36,3,8.33
265,Electronics Item 0265,Electronics,37,3,8.11
